# AMD/NVIDIA SEC 提交历史分卷

从本次官方主目录声明的 2011–2026 相关历史分卷中读取原件，核对两家公司 8-K 年度覆盖及 NAS 备份。

In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_amd_nvda_sec_guidance_original_probe_20261023';raw=root/'data/snapshots/stage1_amd_nvda_sec_guidance_original_probe_20261023';plan=json.loads((card/'older_submissions_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(card/'main_submissions_result.json')==plan['parent_main_result_sha256'] and not (card/'older_submissions_state.json').exists()
state={'status':'started','GET_attempts':0,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'older_submissions_state.json').write_text(json.dumps(state,indent=2)+'\n');receipts=[];checks=[];failure=None
for spec in plan['exact_requests']:
 name=spec['issuer'].lower();body=raw/f'{name}_older_submissions.json';headers=raw/f'{name}_older_submissions.headers';state['GET_attempts']+=1;(card/'older_submissions_state.json').write_text(json.dumps(state,indent=2)+'\n');started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','60','--max-filesize','8000000','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['request']['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=75);parts=run.stdout.rsplit(' ',1);status=int(parts[0]) if len(parts)==2 and parts[0].isdigit() else None
  receipt={'issuer':spec['issuer'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':parts[-1] if parts else None,'stderr':run.stderr[:250]}
 except Exception as exc:receipt={'issuer':spec['issuer'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'exception':str(exc)[:250]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None);receipts.append(receipt);(card/'older_submissions_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists():failure=f'{name}_transport';break
 try:
  j=json.loads(body.read_text());keys=('accessionNumber','filingDate','form','acceptanceDateTime','primaryDocument');n=len(j['accessionNumber']);assert n==spec['expected_catalog_count'] and all(len(j[k])==n for k in keys) and len(set(j['accessionNumber']))==n
  assert min(j['filingDate'])==spec['expected_filing_from'] and max(j['filingDate'])==spec['expected_filing_to']
  main=json.loads((raw/f'{name}_main_submissions.json').read_text())['filings']['recent'];assert not set(j['accessionNumber'])&set(main['accessionNumber'])
  rows=[]
  for source in (j,main):
   rows += [(day,form) for day,form in zip(source['filingDate'],source['form']) if '2011'<=day[:4]<='2026']
  forms=[x for x in rows if x[1]=='8-K'];years={d[:4] for d,f in forms};assert all(str(y) in years for y in range(2011,2027))
  checks.append({'issuer':spec['issuer'],'CIK':spec['CIK'],'raw_sha256':sha(body),'older_rows':n,'combined_2011_2026_8K_count':len(forms),'8K_years_2011_2026':len(years),'older_first_filing':min(j['filingDate']),'older_last_filing':max(j['filingDate'])});(card/'older_submissions_checks.json').write_text(json.dumps(checks,indent=2)+'\n')
 except Exception as exc:failure=f'{name}_schema_{type(exc).__name__}_{str(exc)[:120]}';break
 time.sleep(plan['request']['seconds_between_requests'])
files=[{'name':x.name,'sha256':sha(x),'bytes':x.stat().st_size} for x in sorted(raw.iterdir()) if x.name.endswith(('.json','.headers')) and 'older_submissions' in x.name];manifest={'plan_sha256':sha(card/'older_submissions_plan.json'),'files':files,'receipts_sha256':sha(card/'older_submissions_receipts.json')};(card/'older_submissions_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='amd-nvda-older-') as td:
 archive=Path(td)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in files:t.add(raw/x['name'],arcname='raw/'+x['name'])
  for name in ('older_submissions_plan.json','older_submissions_receipts.json','older_submissions_manifest.json'):t.add(card/name,arcname=name)
 digest=sha(archive);a=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert a['sha256']==digest and a['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='amd-nvda-older-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as t:
   for x in files:
    m=t.extractfile('raw/'+x['name']);assert m and hashlib.sha256(m.read()).hexdigest()==x['sha256']
result={'status':'AMD_NVDA_SEC_8K_metadata_2011_2026_coverage_passed_quarterly_identity_pending' if not failure and len(checks)==2 else 'bounded_AMD_NVDA_SEC_history_partial_or_failed','failure':failure,'plan_sha256':sha(card/'older_submissions_plan.json'),'GET_attempts':state['GET_attempts'],'checks':checks,'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'original_quarterly_guidance_coverage_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'older_submissions_result.json').write_text(json.dumps(result,indent=2)+'\n');state.update(status='completed' if not failure else 'partial_or_failed',NAS_archive_sha256=digest);(card/'older_submissions_state.json').write_text(json.dumps(state,indent=2)+'\n');print({'status':result['status'],'failure':failure,'GETs':state['GET_attempts'],'8Ks':[(x['issuer'],x['combined_2011_2026_8K_count']) for x in checks],'NAS':digest})


{'status': 'bounded_AMD_NVDA_SEC_history_partial_or_failed', 'failure': 'nvda_schema_AssertionError_', 'GETs': 1, '8Ks': [], 'NAS': 'ee70cb8ae038c4e75bfdd9c42e15af9b381b3f361ff5e20fe9818d1ed8e0221e'}


## 中文结论

首次旧目录校验因 NVIDIA 官方声明边界 2020-09-15 与最后实际申报日 09-14 相差一天停止；AMD 未请求，保留部分 NAS 备份。 完整决定见 `conclusion.md`。